# Notebook overview: R8_8_CO_POI_plot.ipynb

## What this notebook does
Creates POI-level maps and exploratory visualizations for observed/counterfactual interaction changes. It loads POI attributes, user-level features, aligned dyads, and POI residual outputs to visualize important venues and flow/interaction patterns.

## Inputs used
- User-level feature table: user_level_feature_table_{revision}.csv
- Aligned dyads: individual_interac_allusers_{revision}_aligned.csv
- POI interaction/residual tables: poi_agg_interac_* and poi_structural_random_merged_*
- Cleaned POI polygons: CO_pois_TP.csv

## Outputs created
- Kepler/interactive map objects
- POI residual visualizations
- Flow and venue-level exploratory plots

**Data access warning.** The Cuebiq/Spectus mobility stop data used by this project are proprietary/restricted and are not included in this repository. Do not commit raw mobility files, user IDs, stop tables, home-location files, graph outputs containing sensitive identifiers, or large derived files unless your data-use agreement explicitly permits release. Public users must obtain data access independently and place files in the documented paths.

In [ ]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
from sklearn.neighbors import BallTree
import numpy as np
from scipy.sparse import lil_matrix
import json
from collections import defaultdict
import networkx as nx
import pickle
import os
import matplotlib.pyplot as plt
from shapely.geometry import Point
from shapely import wkt
#import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from shapely.geometry import LineString, Point

In [ ]:
print("Current working directory:", os.getcwd())
base = os.path.join("..", "Data", "stop_df_perday_CO")
pois_dir = os.path.join(base, "POIs")
geo_dir = os.path.join(base, "geography_CO")
stops_dir = os.path.join(base, "daily_agg_to_weekly_Stops")
graph_poi_dir = os.path.join(base, "graphs", "poi-user")
os.makedirs(graph_poi_dir, exist_ok=True)
week_dir = os.path.join(graph_poi_dir, "user_connections" )
os.makedirs(week_dir, exist_ok=True)
home_dir = os.path.join(base, "home/pre_disaster")
graph_dir = os.path.join(base, "graphs_POI_weighted")
os.makedirs(graph_dir, exist_ok=True)
revision = "R11"

boot_dir = os.path.join(graph_poi_dir, "boots_user_removal", revision)
cf_out_dir = os.path.join(graph_poi_dir, "counterfactual_post", revision)

boot_dir_w = os.path.join(graph_poi_dir, "boots_user_removal_weighted_yhat", revision)
os.makedirs(boot_dir_w, exist_ok=True)

boot_dir_w_count = os.path.join(graph_poi_dir, "boots_user_removal_weighted_yhat_count", revision)
os.makedirs(boot_dir_w_count, exist_ok=True)

os.makedirs(cf_out_dir, exist_ok=True)

survivor_dir = os.path.join(graph_poi_dir, "boots_user_survivors", revision)
os.makedirs(survivor_dir, exist_ok=True)

In [ ]:
out_path = os.path.join(graph_dir, f"user_level_feature_table_{revision}.csv")
uf = pd.read_csv(out_path)
uf.columns


In [ ]:
pip install keplergl

In [ ]:
from keplergl import KeplerGl
from shapely.geometry.base import BaseGeometry

In [ ]:
dyad_path =  os.path.join(base, "for_ABM", f"individual_interac_allusers_{revision}_aligned.csv")
dy = pd.read_csv(dyad_path)
dy.columns

In [ ]:
revision = "R11"
interactions = os.path.join(base, "for_ABM", f"poi_agg_interac_nonevac_{revision}.csv")

poi = pd.read_csv(interactions)
poi.columns



In [ ]:
revision = "R11"
output_path = os.path.join(
    boot_dir_w_count,
    f"poi_structural_random_merged_{revision}_count.csv"
)
poi_z = pd.read_csv(output_path)
poi_z.columns



In [ ]:
pre_edge_df_filename = f"CO_pois_TP.csv"
pre_edge_df_path = os.path.join(pois_dir, pre_edge_df_filename)
pois_MT = pd.read_csv(pre_edge_df_path)
#pre_df["week_phase_label"] = "PDMW1" 
pois_MT.columns

# Old Plotting

In [ ]:
def safe_wkt(x):
    if not isinstance(x, str):
        return None
    x = x.strip()
    if not x:
        return None
    try:
        return wkt.loads(x)
    except Exception:
        return None

# ----------------------------
# 1) Build POI geometry from dy (one row per poi)
# ----------------------------
dy_poi = dy[["poi", "point_geometry", "polygon_geometry", "SUB_CATEGORY"]].drop_duplicates("poi").copy()
dy_poi["poi"] = dy_poi["poi"].astype(str)

dy_poi["pt_geom"]   = dy_poi["point_geometry"].apply(safe_wkt)
dy_poi["poly_geom"] = dy_poi["polygon_geometry"].apply(safe_wkt)

# choose POI point: point_geometry first, else polygon centroid
dy_poi["poi_point"] = None

mask_pt = dy_poi["pt_geom"].apply(lambda g: getattr(g, "geom_type", None) == "Point")
dy_poi.loc[mask_pt, "poi_point"] = dy_poi.loc[mask_pt, "pt_geom"]

mask_poly = dy_poi["poi_point"].isna() & dy_poi["poly_geom"].notna()
dy_poi.loc[mask_poly, "poi_point"] = dy_poi.loc[mask_poly, "poly_geom"].apply(
    lambda g: g.centroid if getattr(g, "geom_type", None) in {"Polygon", "MultiPolygon"} else None
)

# keep only POIs with a valid point
dy_poi = dy_poi.dropna(subset=["poi_point"]).copy()

# extract lat/lng for Kepler (NO shapely in final df)
dy_poi["lng"] = dy_poi["poi_point"].apply(lambda g: float(g.x))
dy_poi["lat"] = dy_poi["poi_point"].apply(lambda g: float(g.y))

# ----------------------------
# 2) Attach pre_sum from poi table
# ----------------------------
poi2 = poi[["poi", "pre_sum"]].copy()
poi2["poi"] = poi2["poi"].astype(str)
poi2["pre_sum"] = pd.to_numeric(poi2["pre_sum"], errors="coerce").fillna(0.0)

poi_df = dy_poi.merge(poi2, on="poi", how="left")
poi_df["pre_sum"] = poi_df["pre_sum"].fillna(0.0)

# keep only POIs with pre interactions
poi_df = poi_df[poi_df["pre_sum"] > 0].copy()

# ----------------------------
# 3) Size + color fields
# ----------------------------
poi_df["log_pre_sum"] = np.log1p(poi_df["pre_sum"])

# FORCE a big visible size spread (rank-based)
poi_df["poi_size"] = 2 + 78 * poi_df["log_pre_sum"].rank(pct=True)   # 2..80

# keep only columns you want on hover
poi_df = poi_df[["poi", "SUB_CATEGORY", "pre_sum", "log_pre_sum", "poi_size", "lat", "lng"]].copy()

# ----------------------------
# 4) Kepler config (lat/lng mode)
# ----------------------------
cfg_poi = {
  "version": "v1",
  "config": {
    "visState": {
      "layers": [
        {
          "id": "poi_points_layer",
          "type": "point",
          "config": {
            "dataId": "POIs",
            "label": "POIs (size=poi_size, color=log_pre_sum)",
            "columns": {"lat": "lat", "lng": "lng"},
            "isVisible": True,
            "visConfig": {
              "opacity": 0.85,
              "filled": True,
              "radiusRange": [2, 50]
            }
          },
          "visualChannels": {
            "sizeField": {"name": "poi_size", "type": "real"},
            "sizeScale": "linear",
            "colorField": {"name": "log_pre_sum", "type": "real"},
            "colorScale": "quantile"
          }
        }
      ]
    }
  }
}

m = KeplerGl(height=750)
m.add_data(data=poi_df, name="POIs")   # name MUST match dataId above
m.config = cfg_poi                    # set AFTER add_data
m

In [ ]:
def safe_wkt(x):
    if not isinstance(x, str):
        return None
    x = x.strip()
    if not x:
        return None
    try:
        return wkt.loads(x)
    except Exception:
        return None

# ----------------------------
# 1) Build POI point per row (geometry only)
# ----------------------------
dy2 = dy[[
    "poi",
    "user_i", "user_j",
    "cbg_centroid_i", "cbg_centroid_j",
    "point_geometry", "polygon_geometry",
    "SUB_CATEGORY",
    "evac_status_i", "evac_status_j"
]].copy()

dy2["poi"]    = dy2["poi"].astype(str)
dy2["user_i"] = dy2["user_i"].astype(str)
dy2["user_j"] = dy2["user_j"].astype(str)

dy2["poi_pt"]   = dy2["point_geometry"].apply(safe_wkt)
dy2["poi_poly"] = dy2["polygon_geometry"].apply(safe_wkt)

# fallback: polygon centroid if point missing
missing_pt = dy2["poi_pt"].isna() & dy2["poi_poly"].notna()
dy2.loc[missing_pt, "poi_pt"] = dy2.loc[missing_pt, "poi_poly"].apply(
    lambda g: g.centroid if getattr(g, "geom_type", None) in {"Polygon", "MultiPolygon"} else None
)

# keep only valid POI points
dy2 = dy2[dy2["poi_pt"].apply(lambda g: getattr(g, "geom_type", None) == "Point")].copy()
dy2["poi_lng"] = dy2["poi_pt"].apply(lambda g: float(g.x))
dy2["poi_lat"] = dy2["poi_pt"].apply(lambda g: float(g.y))

# ----------------------------
# 2) Parse homes (EPSG:4326 lon/lat WKT)
# ----------------------------
dy2["home_i_4326"] = dy2["cbg_centroid_i"].apply(safe_wkt)
dy2["home_j_4326"] = dy2["cbg_centroid_j"].apply(safe_wkt)

flows = pd.concat([
    dy2[["poi","SUB_CATEGORY","poi_lat","poi_lng","user_i","home_i_4326","evac_status_i"]]
      .rename(columns={"user_i":"user","home_i_4326":"home_4326","evac_status_i":"evac_status"}),
    dy2[["poi","SUB_CATEGORY","poi_lat","poi_lng","user_j","home_j_4326","evac_status_j"]]
      .rename(columns={"user_j":"user","home_j_4326":"home_4326","evac_status_j":"evac_status"}),
], ignore_index=True)

flows = flows.dropna(subset=["home_4326"]).copy()
flows = flows[flows["home_4326"].apply(lambda g: getattr(g, "geom_type", None) == "Point")].copy()

home_gdf_4326 = gpd.GeoDataFrame(
    flows[["poi","SUB_CATEGORY","poi_lat","poi_lng","user","evac_status"]].copy(),
    geometry=flows["home_4326"],
    crs="EPSG:4326"
)

home_gdf_4326["home_lng"] = home_gdf_4326.geometry.x
home_gdf_4326["home_lat"] = home_gdf_4326.geometry.y

flows_df = home_gdf_4326.drop(columns="geometry").copy()

# OPTIONAL: dedupe identical user->poi links
flows_df = flows_df.drop_duplicates(subset=["user", "poi"])

# ----------------------------
# 3) Kepler Arc layer
# ----------------------------
cfg_flows = {
  "version": "v1",
  "config": {
    "visState": {
      "layers": [
        {
          "id": "flows_arc",
          "type": "arc",
          "config": {
            "dataId": "Flows",
            "label": "Home → POI links (unique user-poi)",
            "columns": {
              "lat0": "home_lat",
              "lng0": "home_lng",
              "lat1": "poi_lat",
              "lng1": "poi_lng"
            },
            "isVisible": True,
            "visConfig": {"opacity": 0.25}
          }
        }
      ]
    }
  }
}

# 30-ish distinct categorical colors (Tableau/Plotly-like, NOT diverging)
tab30_like = [
    "#1f77b4","#ff7f0e","#2ca02c","#d62728","#9467bd",
    "#8c564b","#e377c2","#7f7f7f","#bcbd22","#17becf",
    "#393b79","#637939","#8c6d31","#843c39","#7b4173",
    "#3182bd","#e6550d","#31a354","#756bb1","#636363",
    "#6baed6","#fd8d3c","#74c476","#9e9ac8","#969696",
    "#9ecae1","#fdae6b","#a1d99b","#bcbddc","#bdbdbd"
]

cfg_flows["config"]["visState"]["layers"][0]["visualChannels"] = {
    "colorField": {"name": "SUB_CATEGORY", "type": "string"},
    "colorScale": "ordinal"
}

cfg_flows["config"]["visState"]["layers"][0]["config"]["visConfig"]["colorRange"] = {
    "name": "tab30_like",
    "type": "qualitative",
    "category": "custom",
    "colors": tab30_like
}

m = KeplerGl(height=750)
m.add_data(data=flows_df, name="Flows")
m.config = cfg_flows
m

In [ ]:
import copy
from keplergl import KeplerGl

# --- 1) build map + add BOTH datasets (use your actual variable names) ---
m = KeplerGl(height=750)
m.add_data(data=flows_df, name="Flows")
m.add_data(data=poi_df,   name="POIs")     # <-- FIX: poi_df is your POI table

# --- 2) start from your working flows config ---
cfg2 = copy.deepcopy(cfg_flows)

# --- 3) add POI point layer (second layer) ---
poi_layer = {
  "id": "poi_points_layer",
  "type": "point",
  "config": {
    "dataId": "POIs",
    "label": "POIs (size=poi_size, color=log_pre_sum)",
    "columns": {"lat": "lat", "lng": "lng"},   # <-- FIX: your POI cols are lat/lng
    "isVisible": True,
    "visConfig": {
      "opacity": 0.85,
      "filled": True,
      "radiusRange": [2, 60]
    }
  },
  "visualChannels": {
    "sizeField": {"name": "poi_size", "type": "real"},
    "sizeScale": "linear",
    "colorField": {"name": "log_pre_sum", "type": "real"},
    "colorScale": "quantile"
  }
}

cfg2["config"]["visState"]["layers"].append(poi_layer)

# --- 4) apply config AFTER add_data ---
m.config = cfg2
m

In [ ]:
def safe_wkt(x):
    if not isinstance(x, str):
        return None
    x = x.strip()
    if not x:
        return None
    try:
        return wkt.loads(x)
    except Exception:
        return None

# ----------------------------
# 1) Build POI polygons from dy (one row per poi)
# ----------------------------
dy_poi = dy[["poi", "polygon_geometry", "SUB_CATEGORY"]].drop_duplicates("poi").copy()
dy_poi["poi"] = dy_poi["poi"].astype(str)

dy_poi["geometry"] = dy_poi["polygon_geometry"].apply(safe_wkt)
dy_poi = dy_poi.dropna(subset=["geometry"]).copy()

# keep only valid polygon-like geometries
dy_poi = dy_poi[dy_poi["geometry"].apply(lambda g: isinstance(g, BaseGeometry))].copy()
dy_poi = dy_poi[dy_poi["geometry"].apply(lambda g: getattr(g, "geom_type", None) in {"Polygon", "MultiPolygon"})].copy()

poi_polys = gpd.GeoDataFrame(dy_poi[["poi", "SUB_CATEGORY"]].copy(), geometry=dy_poi["geometry"], crs="EPSG:4326")

# ----------------------------
# 2) Attach pre_sum + post_sum from poi table
# ----------------------------
poi2 = poi[["poi", "pre_sum", "post_sum"]].copy()
poi2["poi"] = poi2["poi"].astype(str)
poi2["pre_sum"]  = pd.to_numeric(poi2["pre_sum"],  errors="coerce").fillna(0.0)
poi2["post_sum"] = pd.to_numeric(poi2["post_sum"], errors="coerce").fillna(0.0)

poi_polys = poi_polys.merge(poi2, on="poi", how="left")
poi_polys["pre_sum"]  = poi_polys["pre_sum"].fillna(0.0)
poi_polys["post_sum"] = poi_polys["post_sum"].fillna(0.0)

# keep only POIs with pre interactions
poi_polys = poi_polys[poi_polys["pre_sum"] > 0].copy()

# ----------------------------
# 3) Make Kepler-safe GeoJSON geometry column
#    (Kepler likes GeoJSON, not WKT)
# ----------------------------
poi_polys_k = poi_polys[["poi", "SUB_CATEGORY", "pre_sum", "post_sum", "geometry"]].copy()
poi_polys_k["_geojson"] = poi_polys_k["geometry"].apply(lambda g: json.loads(gpd.GeoSeries([g]).to_json())["features"][0]["geometry"])
poi_polys_k = poi_polys_k.drop(columns=["geometry"]).reset_index(drop=True)

vmin = float(poi_polys_k["pre_sum"].min())
vmax = float(poi_polys_k["pre_sum"].max())

# ----------------------------
# 4) Kepler config: polygon fill AND outline both mapped to pre_sum
# ----------------------------
cfg_polys = {
  "version": "v1",
  "config": {
    "visState": {
      "layers": [
        {
          "id": "poi_polygons_layer",
          "type": "geojson",
          "config": {
            "dataId": "POIs (polygons)",
            "label": "POIs (color by pre_sum)",
            "columns": {"geojson": "_geojson"},
            "isVisible": True,
            "visConfig": {
              "opacity": 0.75,
              "thickness": 1,
              "strokeOpacity": 1.0,
              "filled": True
            }
          },
          "visualChannels": {
            "colorField": {"name": "pre_sum", "type": "real"},
            "colorScale": "linear",
            "colorDomain": [vmin, vmax],

            "strokeColorField": {"name": "pre_sum", "type": "real"},
            "strokeColorScale": "linear",
            "strokeColorDomain": [vmin, vmax]
          }
        }
      ],
      "interactionConfig": {
        "tooltip": {
          "enabled": True,
          "fieldsToShow": {
            "POIs (polygons)": ["poi", "SUB_CATEGORY", "pre_sum", "post_sum"]
          }
        }
      }
    }
  }
}

m = KeplerGl(height=750)
m.add_data(data=poi_polys_k, name="POIs (polygons)")
m.config = cfg_polys
m

In [ ]:
def safe_wkt(x):
    if not isinstance(x, str):
        return None
    x = x.strip()
    if not x:
        return None
    try:
        return wkt.loads(x)
    except Exception:
        return None

# ----------------------------
# 1) Build POI polygons from dy (one row per poi)
# ----------------------------
dy_poi = dy[["poi", "polygon_geometry", "SUB_CATEGORY"]].drop_duplicates("poi").copy()
dy_poi["poi"] = dy_poi["poi"].astype(str)

dy_poi["geometry"] = dy_poi["polygon_geometry"].apply(safe_wkt)
dy_poi = dy_poi.dropna(subset=["geometry"]).copy()

dy_poi = dy_poi[dy_poi["geometry"].apply(lambda g: isinstance(g, BaseGeometry))].copy()
dy_poi = dy_poi[dy_poi["geometry"].apply(lambda g: getattr(g, "geom_type", None) in {"Polygon", "MultiPolygon"})].copy()

poi_polys = gpd.GeoDataFrame(
    dy_poi[["poi", "SUB_CATEGORY"]].copy(),
    geometry=dy_poi["geometry"],
    crs="EPSG:4326"
)

# ----------------------------
# 2) Attach pre_sum + post_sum from poi table
# ----------------------------
poi2 = poi[["poi", "pre_sum", "post_sum"]].copy()
poi2["poi"] = poi2["poi"].astype(str)
poi2["pre_sum"]  = pd.to_numeric(poi2["pre_sum"],  errors="coerce").fillna(0.0)
poi2["post_sum"] = pd.to_numeric(poi2["post_sum"], errors="coerce").fillna(0.0)

poi_polys = poi_polys.merge(poi2, on="poi", how="left")
poi_polys["pre_sum"]  = poi_polys["pre_sum"].fillna(0.0)
poi_polys["post_sum"] = poi_polys["post_sum"].fillna(0.0)

poi_polys = poi_polys[poi_polys["pre_sum"] > 0].copy()

# ✅ LOG COLUMN (only new thing)
poi_polys["log_pre_sum"] = np.log1p(poi_polys["pre_sum"])

# ----------------------------
# 3) Make Kepler-safe GeoJSON geometry column
# ----------------------------
poi_polys_k = poi_polys[["poi", "SUB_CATEGORY", "pre_sum", "log_pre_sum", "post_sum", "geometry"]].copy()
poi_polys_k["_geojson"] = poi_polys_k["geometry"].apply(
    lambda g: json.loads(gpd.GeoSeries([g]).to_json())["features"][0]["geometry"]
)
poi_polys_k = poi_polys_k.drop(columns=["geometry"]).reset_index(drop=True)

# ✅ domain on LOG (not raw)
vmin = float(poi_polys_k["log_pre_sum"].min())
vmax = float(poi_polys_k["log_pre_sum"].max())

# ----------------------------
# 4) Kepler config: polygon fill AND outline both mapped to log_pre_sum
# ----------------------------
cfg_polys = {
  "version": "v1",
  "config": {
    "visState": {
      "layers": [
        {
          "id": "poi_polygons_layer",
          "type": "geojson",
          "config": {
            "dataId": "POIs (polygons)",
            "label": "POIs (color by log_pre_sum)",
            "columns": {"geojson": "_geojson"},
            "isVisible": True,
            "visConfig": {
              "opacity": 0.75,
              "thickness": 1,
              "strokeOpacity": 1.0,
              "filled": True
            }
          },
          "visualChannels": {
            "colorField": {"name": "log_pre_sum", "type": "real"},
            "colorScale": "linear",
            "colorDomain": [vmin, vmax],

            "strokeColorField": {"name": "log_pre_sum", "type": "real"},
            "strokeColorScale": "linear",
            "strokeColorDomain": [vmin, vmax]
          }
        }
      ],
      "interactionConfig": {
        "tooltip": {
          "enabled": True,
          "fieldsToShow": {
            "POIs (polygons)": ["poi", "SUB_CATEGORY", "pre_sum", "log_pre_sum", "post_sum"]
          }
        }
      }
    }
  }
}

m = KeplerGl(height=750)
m.add_data(data=poi_polys_k, name="POIs (polygons)")
m.config = cfg_polys
m

In [ ]:
def safe_wkt(x):
    if not isinstance(x, str):
        return None
    x = x.strip()
    if not x:
        return None
    try:
        return wkt.loads(x)
    except Exception:
        return None

dy_poi = dy[["poi", "polygon_geometry", "SUB_CATEGORY"]].drop_duplicates("poi").copy()
dy_poi["poi"] = dy_poi["poi"].astype(str)

dy_poi["geometry"] = dy_poi["polygon_geometry"].apply(safe_wkt)
dy_poi = dy_poi.dropna(subset=["geometry"]).copy()
dy_poi = dy_poi[dy_poi["geometry"].apply(lambda g: isinstance(g, BaseGeometry))].copy()
dy_poi = dy_poi[dy_poi["geometry"].apply(lambda g: getattr(g, "geom_type", None) in {"Polygon", "MultiPolygon"})].copy()

poi_polys = gpd.GeoDataFrame(
    dy_poi[["poi", "SUB_CATEGORY"]].copy(),
    geometry=dy_poi["geometry"],
    crs="EPSG:4326"
)

# ----------------------------
# 2) Attach pre_sum + post_sum from poi table
# ----------------------------
poi2 = poi[["poi", "pre_sum", "post_sum"]].copy()
poi2["poi"] = poi2["poi"].astype(str)
poi2["pre_sum"]  = pd.to_numeric(poi2["pre_sum"],  errors="coerce").fillna(0.0)
poi2["post_sum"] = pd.to_numeric(poi2["post_sum"], errors="coerce").fillna(0.0)

poi_polys = poi_polys.merge(poi2, on="poi", how="left")
poi_polys["pre_sum"]  = poi_polys["pre_sum"].fillna(0.0)
poi_polys["post_sum"] = poi_polys["post_sum"].fillna(0.0)

poi_polys = poi_polys[poi_polys["pre_sum"] > 0].copy()

# ----------------------------
# 3) Attach POI metadata: LOCATION_NAME + STREET_ADDRESS
#    POI id column is PLACEKEY
# ----------------------------
pois_MT2 = (
    pois_MT[["PLACEKEY", "LOCATION_NAME", "STREET_ADDRESS"]]
    .copy()
    .rename(columns={"PLACEKEY": "poi"})
)
pois_MT2["poi"] = pois_MT2["poi"].astype(str)

poi_polys = poi_polys.merge(pois_MT2, on="poi", how="left")

# ----------------------------
# 4) Use LOG pre_sum for color mapping
# ----------------------------
poi_polys["log_pre_sum"] = np.log1p(poi_polys["pre_sum"])

# ----------------------------
# 5) Make Kepler-safe GeoJSON column
# ----------------------------
poi_polys_k = poi_polys[
    ["poi", "LOCATION_NAME", "STREET_ADDRESS", "SUB_CATEGORY",
     "pre_sum", "post_sum", "log_pre_sum", "geometry"]
].copy()

def geom_to_geojson(g):
    return json.loads(gpd.GeoSeries([g], crs="EPSG:4326").to_json())["features"][0]["geometry"]

poi_polys_k["_geojson"] = poi_polys_k["geometry"].apply(geom_to_geojson)
poi_polys_k = poi_polys_k.drop(columns=["geometry"]).reset_index(drop=True)

vmin = float(poi_polys_k["log_pre_sum"].min())
vmax = float(poi_polys_k["log_pre_sum"].max())

# ----------------------------
# 6) Kepler config: polygon fill + outline colored by log_pre_sum
# ----------------------------
cfg_polys = {
  "version": "v1",
  "config": {
    "visState": {
      "layers": [
        {
          "id": "poi_polygons_layer",
          "type": "geojson",
          "config": {
            "dataId": "POIs (polygons)",
            "label": "POIs (color by log(pre_sum))",
            "columns": {"geojson": "_geojson"},
            "isVisible": True,
            "visConfig": {
              "opacity": 0.80,
              "thickness": 1,
              "strokeOpacity": 1.0,
              "filled": True
            }
          },
          "visualChannels": {
            "colorField": {"name": "log_pre_sum", "type": "real"},
            "colorScale": "quantile",
            "colorDomain": [vmin, vmax],
            "strokeColorField": {"name": "log_pre_sum", "type": "real"},
            "strokeColorScale": "quantile",
            "strokeColorDomain": [vmin, vmax]
          }
        }
      ],
      "interactionConfig": {
        "tooltip": {
          "enabled": True,
          "fieldsToShow": {
            "POIs (polygons)": [
              "poi", "LOCATION_NAME", "STREET_ADDRESS",
              "SUB_CATEGORY", "pre_sum", "post_sum", "log_pre_sum"
            ]
          }
        }
      }
    }
  }
}

m = KeplerGl(height=750)
m.add_data(data=poi_polys_k, name="POIs (polygons)")
m.config = cfg_polys
m

# Final Plot

In [ ]:
df = poi.copy()

df["pre_sum"]  = pd.to_numeric(df["pre_sum"], errors="coerce").fillna(0.0)
df["post_sum"] = pd.to_numeric(df["post_sum"], errors="coerce").fillna(0.0)

df["post_pre_ratio"] = np.where(df["pre_sum"] > 0, df["post_sum"] / df["pre_sum"], np.nan)

df_hi = df[
    (df["pre_sum"] > 0) &
    (df["post_sum"] >= 50) &
    (df["post_sum"] > 1.5 * df["pre_sum"])
].copy()


print("Selected POIs:", len(df_hi))

# --- meta columns you want ---
meta_cols = ["PLACEKEY", "LOCATION_NAME", "STREET_ADDRESS"]

pois_meta = pois_MT[meta_cols].copy()
pois_meta["PLACEKEY"] = pois_meta["PLACEKEY"].astype(str)

# --- make sure poi id is string + same key name ---
df_hi = df_hi.copy()
df_hi["poi"] = df_hi["poi"].astype(str)

# --- merge ---
df_hi = df_hi.merge(
    pois_meta.rename(columns={"PLACEKEY": "poi"}),
    on="poi",
    how="left"
)

# optional: reorder for readability
front = ["poi", "LOCATION_NAME", "STREET_ADDRESS", "pre_sum", "post_sum", "post_pre_ratio"]
rest = [c for c in df_hi.columns if c not in front]
df_hi = df_hi[front + rest]

df_hi[["poi","pre_sum","post_sum","post_pre_ratio","TOP_CATEGORY","SUB_CATEGORY" , "LOCATION_NAME", "STREET_ADDRESS",]].head()

In [ ]:
def safe_wkt(x):
    if not isinstance(x, str):
        return None
    x = x.strip()
    if not x:
        return None
    try:
        return wkt.loads(x)
    except Exception:
        return None

dy_poi = dy[["poi", "polygon_geometry", "SUB_CATEGORY"]].drop_duplicates("poi").copy()
dy_poi["poi"] = dy_poi["poi"].astype(str)

dy_poi["geometry"] = dy_poi["polygon_geometry"].apply(safe_wkt)
dy_poi = dy_poi.dropna(subset=["geometry"]).copy()
dy_poi = dy_poi[dy_poi["geometry"].apply(lambda g: isinstance(g, BaseGeometry))].copy()
dy_poi = dy_poi[dy_poi["geometry"].apply(lambda g: getattr(g, "geom_type", None) in {"Polygon", "MultiPolygon"})].copy()

poi_polys = gpd.GeoDataFrame(
    dy_poi[["poi", "SUB_CATEGORY"]].copy(),
    geometry=dy_poi["geometry"],
    crs="EPSG:4326"
)

# ----------------------------
# 2) Attach pre_sum + post_sum from poi table
# ----------------------------
poi2 = poi[["poi", "pre_sum", "post_sum"]].copy()
poi2["poi"] = poi2["poi"].astype(str)
poi2["pre_sum"]  = pd.to_numeric(poi2["pre_sum"],  errors="coerce").fillna(0.0)
poi2["post_sum"] = pd.to_numeric(poi2["post_sum"], errors="coerce").fillna(0.0)

poi_polys = poi_polys.merge(poi2, on="poi", how="left")
poi_polys["pre_sum"]  = poi_polys["pre_sum"].fillna(0.0)
poi_polys["post_sum"] = poi_polys["post_sum"].fillna(0.0)

poi_polys = poi_polys[poi_polys["pre_sum"] > 0].copy()

# ----------------------------
# 3) Attach POI metadata: LOCATION_NAME + STREET_ADDRESS
#    POI id column is PLACEKEY
# ----------------------------
pois_MT2 = (
    pois_MT[["PLACEKEY", "LOCATION_NAME", "STREET_ADDRESS"]]
    .copy()
    .rename(columns={"PLACEKEY": "poi"})
)
pois_MT2["poi"] = pois_MT2["poi"].astype(str)

poi_polys = poi_polys.merge(pois_MT2, on="poi", how="left")

# ----------------------------
# 4) Use LOG pre_sum for color mapping
# ----------------------------
poi_polys["log_pre_sum"] = np.log1p(poi_polys["pre_sum"])

# ----------------------------
# 5) Make Kepler-safe GeoJSON column
# ----------------------------
poi_polys_k = poi_polys[
    ["poi", "LOCATION_NAME", "STREET_ADDRESS", "SUB_CATEGORY",
     "pre_sum", "post_sum", "log_pre_sum", "geometry"]
].copy()

def geom_to_geojson(g):
    return json.loads(gpd.GeoSeries([g], crs="EPSG:4326").to_json())["features"][0]["geometry"]

poi_polys_k["_geojson"] = poi_polys_k["geometry"].apply(geom_to_geojson)
poi_polys_k = poi_polys_k.drop(columns=["geometry"]).reset_index(drop=True)

vmin = float(poi_polys_k["log_pre_sum"].min())
vmax = float(poi_polys_k["log_pre_sum"].max())

# --- 0) build a second dataset with ONLY hi-post polygons ---
hi_set = set(df_hi["poi"].astype(str))
poi_polys_k = poi_polys_k.copy()
poi_polys_k["is_hi_post"] = poi_polys_k["poi"].astype(str).isin(hi_set)

poi_hi_k = poi_polys_k.loc[poi_polys_k["is_hi_post"]].copy()

# --- 1) Kepler config: base layer (colored by log_pre_sum) + highlight layer (white outline) ---
cfg_polys = {
  "version": "v1",
  "config": {
    "visState": {
      "layers": [
        # Base polygons colored by log_pre_sum
        {
          "id": "poi_polygons_layer",
          "type": "geojson",
          "config": {
            "dataId": "POIs (polygons)",
            "label": "POIs (color by log(pre_sum))",
            "columns": {"geojson": "_geojson"},
            "isVisible": True,
            "visConfig": {
              "opacity": 0.80,
              "thickness": 1,
              "strokeOpacity": 1.0,
              "filled": True
            }
          },
          "visualChannels": {
            "colorField": {"name": "log_pre_sum", "type": "real"},
            "colorScale": "quantile",
            "strokeColorField": {"name": "log_pre_sum", "type": "real"},
            "strokeColorScale": "quantile"
          }
        },

        # Highlight: ONLY hi-post polygons, thick WHITE outline
        {
          "id": "poi_hi_outline",
          "type": "geojson",
          "config": {
            "dataId": "POIs_hi_post",
            "label": "High post_sum (white outline)",
            "columns": {"geojson": "_geojson"},
            "isVisible": True,
            "visConfig": {
              "filled": False,
              "opacity": 0.0,
              "strokeOpacity": 1.0,
              "thickness": 2,
              "strokeColor": [255, 255, 255]   # <-- this is the key
            }
          },
          "visualChannels": {}
        }
      ],
      "interactionConfig": {
        "tooltip": {
          "enabled": True,
          "fieldsToShow": {
            "POIs (polygons)": [
              "poi", "LOCATION_NAME", "STREET_ADDRESS",
              "SUB_CATEGORY", "pre_sum", "post_sum", "log_pre_sum"
            ],
            "POIs_hi_post": [
              "poi", "LOCATION_NAME", "STREET_ADDRESS",
              "SUB_CATEGORY", "pre_sum", "post_sum"
            ]
          }
        }
      }
    }
  }
}

# --- 2) Build map (IMPORTANT: add BOTH datasets, then set config) ---
from keplergl import KeplerGl

m = KeplerGl(height=750)
m.add_data(data=poi_polys_k, name="POIs (polygons)")
m.add_data(data=poi_hi_k,   name="POIs_hi_post")
m.config = cfg_polys
m

In [ ]:
def safe_wkt(x):
    if not isinstance(x, str):
        return None
    x = x.strip()
    if not x:
        return None
    try:
        return wkt.loads(x)
    except Exception:
        return None

# ----------------------------
# 1) Build POI polygons from dy (one row per poi)
# ----------------------------
dy_poi = dy[["poi", "polygon_geometry", "SUB_CATEGORY"]].drop_duplicates("poi").copy()
dy_poi["poi"] = dy_poi["poi"].astype(str)

dy_poi["geometry"] = dy_poi["polygon_geometry"].apply(safe_wkt)
dy_poi = dy_poi.dropna(subset=["geometry"]).copy()

dy_poi = dy_poi[dy_poi["geometry"].apply(lambda g: isinstance(g, BaseGeometry))].copy()
dy_poi = dy_poi[dy_poi["geometry"].apply(lambda g: getattr(g, "geom_type", None) in {"Polygon", "MultiPolygon"})].copy()

poi_polys = gpd.GeoDataFrame(
    dy_poi[["poi", "SUB_CATEGORY"]].copy(),
    geometry=dy_poi["geometry"],
    crs="EPSG:4326"
)

# ----------------------------
# 2) Attach pre_sum + post_sum from poi table
# ----------------------------
poi2 = poi[["poi", "pre_sum", "post_sum"]].copy()
poi2["poi"] = poi2["poi"].astype(str)
poi2["pre_sum"]  = pd.to_numeric(poi2["pre_sum"],  errors="coerce").fillna(0.0)
poi2["post_sum"] = pd.to_numeric(poi2["post_sum"], errors="coerce").fillna(0.0)

poi_polys = poi_polys.merge(poi2, on="poi", how="left")
poi_polys["pre_sum"]  = poi_polys["pre_sum"].fillna(0.0)
poi_polys["post_sum"] = poi_polys["post_sum"].fillna(0.0)

poi_polys = poi_polys[poi_polys["pre_sum"] > 0].copy()

# ----------------------------
# 3) Attach POI metadata: LOCATION_NAME + STREET_ADDRESS
#    POI id column is PLACEKEY
# ----------------------------
pois_MT2 = (
    pois_MT[["PLACEKEY", "LOCATION_NAME", "STREET_ADDRESS"]]
    .copy()
    .rename(columns={"PLACEKEY": "poi"})
)
pois_MT2["poi"] = pois_MT2["poi"].astype(str)

poi_polys = poi_polys.merge(pois_MT2, on="poi", how="left")

# ----------------------------
# 4) Attach z, z_log from poi_z  (NEW)
# ----------------------------
poi_z2 = poi_z[["poi", "z", "z_log"]].copy()
poi_z2["poi"] = poi_z2["poi"].astype(str)
poi_z2["z"] = pd.to_numeric(poi_z2["z"], errors="coerce")
poi_z2["z_log"] = pd.to_numeric(poi_z2["z_log"], errors="coerce")

poi_polys = poi_polys.merge(poi_z2, on="poi", how="left")

# ----------------------------
# 5) Use LOG pre_sum for color mapping
# ----------------------------
poi_polys["log_pre_sum"] = np.log1p(poi_polys["pre_sum"])

# ----------------------------
# 6) Make Kepler-safe GeoJSON column
# ----------------------------
poi_polys_k = poi_polys[
    ["poi", "LOCATION_NAME", "STREET_ADDRESS", "SUB_CATEGORY",
     "pre_sum", "post_sum", "log_pre_sum", "z", "z_log", "geometry"]
].copy()

def geom_to_geojson(g):
    return json.loads(gpd.GeoSeries([g], crs="EPSG:4326").to_json())["features"][0]["geometry"]

poi_polys_k["_geojson"] = poi_polys_k["geometry"].apply(geom_to_geojson)
poi_polys_k = poi_polys_k.drop(columns=["geometry"]).reset_index(drop=True)

vmin = float(poi_polys_k["log_pre_sum"].min())
vmax = float(poi_polys_k["log_pre_sum"].max())

# reverse the *values* (not the palette)
poi_polys_k["log_pre_sum_rev"] = vmax - poi_polys_k["log_pre_sum"]
vmin_r = float(poi_polys_k["log_pre_sum_rev"].min())
vmax_r = float(poi_polys_k["log_pre_sum_rev"].max())

# ----------------------------
# 7) Kepler config: polygon fill + outline colored by log_pre_sum
# ----------------------------
cfg_polys = {
  "version": "v1",
  "config": {
    "visState": {
      "layers": [
        {
          "id": "poi_polygons_layer",
          "type": "geojson",
          "config": {
            "dataId": "POIs (polygons)",
            "label": "POIs (color by log(pre_sum))",
            "columns": {"geojson": "_geojson"},
            "isVisible": True,
            "visConfig": {
              "opacity": 0.80,
              "thickness": 1,
              "strokeOpacity": 1.0,
              "filled": True
            }
          },
          "visualChannels": {
            "colorField": {"name": "log_pre_sum_rev", "type": "real"},
            "colorDomain": [vmin_r, vmax_r],
            "strokeColorField": {"name": "log_pre_sum_rev", "type": "real"},
            "strokeColorDomain": [vmin_r, vmax_r],
            "colorScale": "quantile",
            "strokeColorScale": "quantile",
          }
        }
      ],
      "interactionConfig": {
        "tooltip": {
          "enabled": True,
          "fieldsToShow": {
            "POIs (polygons)": [
              "poi", "LOCATION_NAME", "STREET_ADDRESS",
              "SUB_CATEGORY", "pre_sum", "post_sum", "log_pre_sum",
              "z", "z_log"
            ]
          }
        }
      }
    }
  }
}
# --- reverse mapping WITHOUT changing palette ---
poi_polys_k["log_pre_sum_rev"] = vmax - poi_polys_k["log_pre_sum"]
vmin_r = float(poi_polys_k["log_pre_sum_rev"].min())
vmax_r = float(poi_polys_k["log_pre_sum_rev"].max())

layer = cfg_polys["config"]["visState"]["layers"][0]["visualChannels"]

layer["colorField"] = {"name": "log_pre_sum_rev", "type": "real"}
layer["colorDomain"] = [vmin_r, vmax_r]

layer["strokeColorField"] = {"name": "log_pre_sum_rev", "type": "real"}
layer["strokeColorDomain"] = [vmin_r, vmax_r]

m = KeplerGl(height=750)
m.add_data(data=poi_polys_k, name="POIs (polygons)")
m.config = cfg_polys
m

In [ ]:
def safe_wkt(x):
    if not isinstance(x, str):
        return None
    x = x.strip()
    if not x:
        return None
    try:
        return wkt.loads(x)
    except Exception:
        return None

# ----------------------------
# 1) Build POI polygons from dy (one row per poi)
# ----------------------------
dy_poi = dy[["poi", "polygon_geometry", "SUB_CATEGORY"]].drop_duplicates("poi").copy()
dy_poi["poi"] = dy_poi["poi"].astype(str)

dy_poi["geometry"] = dy_poi["polygon_geometry"].apply(safe_wkt)
dy_poi = dy_poi.dropna(subset=["geometry"]).copy()

dy_poi = dy_poi[dy_poi["geometry"].apply(lambda g: isinstance(g, BaseGeometry))].copy()
dy_poi = dy_poi[dy_poi["geometry"].apply(lambda g: getattr(g, "geom_type", None) in {"Polygon", "MultiPolygon"})].copy()

poi_polys = gpd.GeoDataFrame(
    dy_poi[["poi", "SUB_CATEGORY"]].copy(),
    geometry=dy_poi["geometry"],
    crs="EPSG:4326"
)

# ----------------------------
# 2) Attach pre_sum + post_sum from poi table
# ----------------------------
poi2 = poi[["poi", "pre_sum", "post_sum"]].copy()
poi2["poi"] = poi2["poi"].astype(str)
poi2["pre_sum"]  = pd.to_numeric(poi2["pre_sum"],  errors="coerce").fillna(0.0)
poi2["post_sum"] = pd.to_numeric(poi2["post_sum"], errors="coerce").fillna(0.0)

poi_polys = poi_polys.merge(poi2, on="poi", how="left")
poi_polys["pre_sum"]  = poi_polys["pre_sum"].fillna(0.0)
poi_polys["post_sum"] = poi_polys["post_sum"].fillna(0.0)

poi_polys = poi_polys[poi_polys["pre_sum"] > 0].copy()

# ----------------------------
# 3) Attach POI metadata: LOCATION_NAME + STREET_ADDRESS
#    POI id column is PLACEKEY
# ----------------------------
pois_MT2 = (
    pois_MT[["PLACEKEY", "LOCATION_NAME", "STREET_ADDRESS"]]
    .copy()
    .rename(columns={"PLACEKEY": "poi"})
)
pois_MT2["poi"] = pois_MT2["poi"].astype(str)

poi_polys = poi_polys.merge(pois_MT2, on="poi", how="left")

# ----------------------------
# 4) Attach z, z_log from poi_z
# ----------------------------
poi_z2 = poi_z[["poi", "z", "z_log"]].copy()
poi_z2["poi"] = poi_z2["poi"].astype(str)
poi_z2["z"] = pd.to_numeric(poi_z2["z"], errors="coerce")
poi_z2["z_log"] = pd.to_numeric(poi_z2["z_log"], errors="coerce")

poi_polys = poi_polys.merge(poi_z2, on="poi", how="left")

# ----------------------------
# 5) USE z_log for color mapping (ONLY CHANGE)
#    Recompute z_log safely from z, then reverse values (not palette)
# ----------------------------
poi_polys["z"] = pd.to_numeric(poi_polys["z"], errors="coerce")

# z can be negative; use log1p of |z| to keep it finite and monotone in magnitude
poi_polys["z_log_clean"] = np.log1p(np.abs(poi_polys["z"]))

# drop any remaining bad rows
poi_polys = poi_polys.replace([np.inf, -np.inf], np.nan).dropna(subset=["z_log_clean"]).copy()

zmin = float(poi_polys["z_log_clean"].min())
zmax = float(poi_polys["z_log_clean"].max())

poi_polys["z_log_rev"] = zmax - poi_polys["z_log_clean"]

zmin_r = float(poi_polys["z_log_rev"].min())
zmax_r = float(poi_polys["z_log_rev"].max())

# ----------------------------
# 6) Make Kepler-safe GeoJSON column
# ----------------------------
poi_polys_k = poi_polys[
    ["poi", "LOCATION_NAME", "STREET_ADDRESS", "SUB_CATEGORY",
     "pre_sum", "post_sum", "z", "z_log", "z_log_rev", "geometry"]
].copy()

def geom_to_geojson(g):
    return json.loads(gpd.GeoSeries([g], crs="EPSG:4326").to_json())["features"][0]["geometry"]

poi_polys_k["_geojson"] = poi_polys_k["geometry"].apply(geom_to_geojson)
poi_polys_k = poi_polys_k.drop(columns=["geometry"]).reset_index(drop=True)
poi_polys_k
# ----------------------------
# 7) Kepler config: polygon fill + outline colored by z_log_rev
#    (palette unchanged; quantile scale as before)
# ----------------------------
cfg_polys = {
  "version": "v1",
  "config": {
    "visState": {
      "layers": [
        {
          "id": "poi_polygons_layer",
          "type": "geojson",
          "config": {
            "dataId": "POIs (polygons)",
            "label": "POIs (color by z_log)",
            "columns": {"geojson": "_geojson"},
            "isVisible": True,
            "visConfig": {
              "opacity": 0.80,
              "thickness": 1,
              "strokeOpacity": 1.0,
              "filled": True
            }
          },
          "visualChannels": {
            "colorField": {"name": "z_log_rev", "type": "real"},
            "colorScale": "quantile",
            "colorDomain": [zmin_r, zmax_r],
            "strokeColorField": {"name": "z_log_rev", "type": "real"},
            "strokeColorScale": "quantile",
            "strokeColorDomain": [zmin_r, zmax_r]
          }
        }
      ],
      "interactionConfig": {
        "tooltip": {
          "enabled": True,
          "fieldsToShow": {
            "POIs (polygons)": [
              "poi", "LOCATION_NAME", "STREET_ADDRESS",
              "SUB_CATEGORY", "pre_sum", "post_sum",
              "z", "z_log"
            ]
          }
        }
      }
    }
  }
}

m = KeplerGl(height=750)
m.add_data(data=poi_polys_k, name="POIs (polygons)")
m.config = cfg_polys
m

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import json
from shapely import wkt
from shapely.geometry.base import BaseGeometry
from keplergl import KeplerGl

def safe_wkt(x):
    if not isinstance(x, str):
        return None
    x = x.strip()
    if not x:
        return None
    try:
        return wkt.loads(x)
    except Exception:
        return None

def geom_to_geojson(g):
    return json.loads(gpd.GeoSeries([g], crs="EPSG:4326").to_json())["features"][0]["geometry"]

# ============================================================
# 1) Build POI polygons from dy (one row per poi)
# ============================================================
dy_poi = dy[["poi", "polygon_geometry", "SUB_CATEGORY"]].drop_duplicates("poi").copy()
dy_poi["poi"] = dy_poi["poi"].astype(str)

dy_poi["geometry"] = dy_poi["polygon_geometry"].apply(safe_wkt)
dy_poi = dy_poi.dropna(subset=["geometry"]).copy()
dy_poi = dy_poi[dy_poi["geometry"].apply(lambda g: isinstance(g, BaseGeometry))].copy()
dy_poi = dy_poi[dy_poi["geometry"].apply(lambda g: getattr(g, "geom_type", None) in {"Polygon", "MultiPolygon"})].copy()

poi_polys = gpd.GeoDataFrame(
    dy_poi[["poi", "SUB_CATEGORY"]].copy(),
    geometry=dy_poi["geometry"],
    crs="EPSG:4326"
)

# ============================================================
# 2) Attach pre_sum + post_sum from poi table
# ============================================================
poi2 = poi[["poi", "pre_sum", "post_sum"]].copy()
poi2["poi"] = poi2["poi"].astype(str)
poi2["pre_sum"]  = pd.to_numeric(poi2["pre_sum"],  errors="coerce").fillna(0.0)
poi2["post_sum"] = pd.to_numeric(poi2["post_sum"], errors="coerce").fillna(0.0)

poi_polys = poi_polys.merge(poi2, on="poi", how="left")
poi_polys["pre_sum"]  = poi_polys["pre_sum"].fillna(0.0)
poi_polys["post_sum"] = poi_polys["post_sum"].fillna(0.0)

poi_polys = poi_polys[poi_polys["pre_sum"] > 0].copy()

# ============================================================
# 3) Attach POI metadata: LOCATION_NAME + STREET_ADDRESS
#    POI id column is PLACEKEY
# ============================================================
pois_MT2 = (
    pois_MT[["PLACEKEY", "LOCATION_NAME", "STREET_ADDRESS"]]
    .copy()
    .rename(columns={"PLACEKEY": "poi"})
)
pois_MT2["poi"] = pois_MT2["poi"].astype(str)

poi_polys = poi_polys.merge(pois_MT2, on="poi", how="left")

# ============================================================
# 4) Attach z, z_log from poi_z
# ============================================================
poi_z2 = poi_z[["poi", "z", "z_log"]].copy()
poi_z2["poi"] = poi_z2["poi"].astype(str)
poi_z2["z"] = pd.to_numeric(poi_z2["z"], errors="coerce")
poi_z2["z_log"] = pd.to_numeric(poi_z2["z_log"], errors="coerce")

poi_polys = poi_polys.merge(poi_z2, on="poi", how="left")

# ============================================================
# 5) Build log fields + reversed fields (keep palette unchanged)
#    - PRE layer: reverse via log_pre_sum_rev
#    - Z layer: reverse via z_log_rev
# ============================================================
poi_polys["log_pre_sum"] = np.log1p(poi_polys["pre_sum"])
pmax = float(poi_polys["log_pre_sum"].max())
poi_polys["log_pre_sum_rev"] = pmax - poi_polys["log_pre_sum"]

poi_polys["z"] = pd.to_numeric(poi_polys["z"], errors="coerce")
poi_polys["z_log_clean"] = np.log1p(np.abs(poi_polys["z"]))
poi_polys = poi_polys.replace([np.inf, -np.inf], np.nan).dropna(subset=["z_log_clean"]).copy()

zmax = float(poi_polys["z_log_clean"].max())
poi_polys["z_log_rev"] = zmax - poi_polys["z_log_clean"]

# ============================================================
# 6) Kepler-safe dataframe with _geojson
# ============================================================
poi_polys_k = poi_polys[
    ["poi", "LOCATION_NAME", "STREET_ADDRESS", "SUB_CATEGORY",
     "pre_sum", "post_sum",
     "log_pre_sum", "log_pre_sum_rev",
     "z", "z_log", "z_log_clean", "z_log_rev",
     "geometry"]
].copy()

poi_polys_k["_geojson"] = poi_polys_k["geometry"].apply(geom_to_geojson)
poi_polys_k = poi_polys_k.drop(columns=["geometry"]).reset_index(drop=True)

pmin_r = float(poi_polys_k["log_pre_sum_rev"].min())
pmax_r = float(poi_polys_k["log_pre_sum_rev"].max())
zmin_r = float(poi_polys_k["z_log_rev"].min())
zmax_r = float(poi_polys_k["z_log_rev"].max())

# ============================================================
# 7) Two layers: PRE (reversed log_pre_sum) + Z (reversed z_log)
# ============================================================
cfg_two_layers = {
  "version": "v1",
  "config": {
    "visState": {
      "layers": [
        {
          "id": "poi_polygons_pre",
          "type": "geojson",
          "config": {
            "dataId": "POIs (polygons)",
            "label": "POIs (color by log(pre_sum), reversed)",
            "columns": {"geojson": "_geojson"},
            "isVisible": True,
            "visConfig": {
              "opacity": 0.80,
              "thickness": 1,
              "strokeOpacity": 1.0,
              "filled": True
            }
          },
          "visualChannels": {
            "colorField": {"name": "log_pre_sum_rev", "type": "real"},
            "colorScale": "quantile",
            "colorDomain": [pmin_r, pmax_r],
            "strokeColorField": {"name": "log_pre_sum_rev", "type": "real"},
            "strokeColorScale": "quantile",
            "strokeColorDomain": [pmin_r, pmax_r],
          }
        },
        {
          "id": "poi_polygons_z",
          "type": "geojson",
          "config": {
            "dataId": "POIs (polygons)",
            "label": "POIs (color by z_log, reversed)",
            "columns": {"geojson": "_geojson"},
            "isVisible": False,
            "visConfig": {
              "opacity": 0.80,
              "thickness": 1,
              "strokeOpacity": 1.0,
              "filled": True
            }
          },
          "visualChannels": {
            "colorField": {"name": "z_log_rev", "type": "real"},
            "colorScale": "quantile",
            "colorDomain": [zmin_r, zmax_r],
            "strokeColorField": {"name": "z_log_rev", "type": "real"},
            "strokeColorScale": "quantile",
            "strokeColorDomain": [zmin_r, zmax_r],
          }
        }
      ],
      "interactionConfig": {
        "tooltip": {
          "enabled": True,
          "fieldsToShow": {
            "POIs (polygons)": [
              "poi", "LOCATION_NAME", "STREET_ADDRESS",
              "SUB_CATEGORY", "pre_sum", "post_sum",
              "log_pre_sum", "z", "z_log"
            ]
          }
        }
      }
    }
  }
}

m = KeplerGl(height=750)
m.add_data(data=poi_polys_k, name="POIs (polygons)")
m.config = cfg_two_layers
m

In [ ]:
# Save as a standalone interactive HTML
html_path = os.path.join(base, f"kepler_pois_polygons_{revision}.html")
m.save_to_html(file_name=html_path, read_only=True)
print("Saved:", html_path)